# Multi-Provider LLM Integration Guide
*Unified Approaches for Calling LLMs Across different Providers*

*Last updated: March 8, 2026*

This notebook demonstrates multi-provider approaches to calling LLMs, allowing easy switching between providers without code changes. These methods provide flexibility and prevent vendor lock-in.

## Quick Comparison Table

| Method | Multi-Provider? | Streaming? | Local Support | Cost |
|--------|-----------------|------------|---------------|------|
| LiteLLM | Yes | Yes | Via API/Local | Varies |
| LangChain | Yes | Yes | Via integrations (Ollama/HF) | Varies |
| OpenAI Client (Base URLs) | Partial | Yes | Via Ollama | Varies |

## LiteLLM API
Unified API across providers using prefixed model names (e.g., "openai/gpt-4o"). Wide-compatibility with providers.

- **Pros**: Single interface for many providers; easy switching.  
- **Cons**: Extra dependency; some features may vary.  
- **Use Case**: Prototyping across providers without code changes.
- **Documentation Links**: [LiteLLM](https://docs.litellm.ai/)

In [1]:
# imports
from dotenv import load_dotenv
load_dotenv(override=True)
import litellm

# prepare message
message = "What is the recommended max length of power nap?"
messages = [{"role": "user", "content": message}]

# simple test call to openai
openai_response = litellm.completion(model="openai/gpt-4o", messages=messages)
print("\n# openai response:")
print(openai_response.choices[0].message.content)

# simple test call to gemini
# gemini_response = litellm.completion(model="gemini/gemini-3.1-flash-lite-preview", messages=messages)
# print("\n# gemini response:")
# print(gemini_response.choices[0].message.content)

# simple test call to ollama
llama_response = litellm.completion(model="ollama/llama3.2:1b", messages=messages)
print("\n# ollama response:")
print(llama_response.choices[0].message.content)

## LangChain Framework
Framework-based integration using chat model classes for structured LLM calls.
Choose LangChain if you need higher-level orchestration (chains, tools, memory). Wide-compatibility with providers, including easy HuggingFace integration.

- **Pros**: Rich ecosystem for agents/chains; seamless provider swaps.  
- **Cons**: Steeper learning curve; more overhead.  
- **Use Case**: Building complex LLM applications with memory/tools.
- **Documentation Links**: [LangChain](https://python.langchain.com/)

In [2]:
# imports
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# prepare message
message = "What is the recommended max length of power nap?"

# simple test call to openai
openai_model = ChatOpenAI(model="gpt-4o")
openai_response = openai_model.invoke([HumanMessage(content=message)])
print("\n# openai response:")
print(openai_response.content)

# simple test call to gemini
# gemini_model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")
# gemini_response = gemini_model.invoke([HumanMessage(content=message)])
# print("\n# gemini response:")
# print(gemini_response.content)

# simple test call to ollama
ollama_model = ChatOllama(model="llama3.2:1b")
ollama_response = ollama_model.invoke([HumanMessage(content=message)])
print("\n# ollama response:")
print(ollama_response.content)

## OpenAI Client with Base URLs
Uses the OpenAI client with custom base URLs for compatible providers. Natively from OpenAI, but compatible with other providers also. This is the lightest multi-provider approach. Supports local models via Ollama's OpenAI-compatible API.

- **Pros**: Familiar OpenAI interface; lightweight; includes local Ollama support.  
- **Cons**: Limited to OpenAI-compatible endpoints.  
- **Use Case**: Quick multi-provider setup with minimal changes, including local inference.
- **Documentation Links**: [OpenAI](https://platform.openai.com/docs).

In [3]:
# imports

import os
from dotenv import load_dotenv
load_dotenv(override=True)
from openai import OpenAI

google_api_key = os.getenv('GOOGLE_API_KEY')

# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints
openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_url)
ollama_model = "llama3.2:1b"

# prepare message
message = "What is the recommended max length of power nap?"
messages = [{"role": "user", "content": message}]

# simple test call to openai
openai_response = openai.chat.completions.create(model="gpt-4o", messages=messages)
print("\n# openai response:")
print(openai_response.choices[0].message.content)

# simple test call to gemini
# gemini_response = gemini.chat.completions.create(model="gemini-3.1-flash-lite-preview", messages=messages)
# print("\n# gemini response:")
# print(gemini_response.choices[0].message.content)

# simple test call to ollama
ollama_response = ollama.chat.completions.create(model=ollama_model, messages=messages)
print("\n# ollama response:")
print(ollama_response.choices[0].message.content)